In [1]:
# =========================
# 0. Import Libraries
# =========================

import sys
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# =========================
# 1. Setup Project Path
# =========================

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\2.DAP391m_Project


In [5]:
# =========================
# 2. Import Project Config
# =========================

from src.config import (
    AB_DATA_PATH,
    COUNTRY_DATA_PATH,
    CLEANED_DATA_PATH,
    RESULTS_DIR,
    FIGURES_DIR
)

print("AB data path:", AB_DATA_PATH)
print("Country data path:", COUNTRY_DATA_PATH)

print("AB file exists:", AB_DATA_PATH.exists())
print("Country file exists:", COUNTRY_DATA_PATH.exists())

ImportError: cannot import name 'AB_DATA_PATH' from 'src.config' (d:\2.DAP391m_Project\src\config.py)

In [ ]:
# =========================
# 3. Load Raw Data
# =========================

ab_df = pd.read_csv(AB_DATA_PATH)
country_df = pd.read_csv(COUNTRY_DATA_PATH)

print("Import CSV files successfully!")
print("A/B data shape:", ab_df.shape)
print("Country data shape:", country_df.shape)

display(ab_df.head())
display(country_df.head())

Import CSV files successfully!
A/B data shape: (294480, 5)
Country data shape: (290586, 2)


,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


,user_id,country
0,834778,UK
1,928468,US
2,822059,UK
3,711597,UK
4,710616,UK


In [ ]:
#Basic Data Inspection
ab_df.shape
country_df.shape

ab_df.head()
country_df.head()

ab_df.info()
country_df.info()

ab_df.columns
country_df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294480 entries, 0 to 294479
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294480 non-null  int64 
 1   timestamp     294480 non-null  object
 2   group         294480 non-null  object
 3   landing_page  294480 non-null  object
 4   converted     294480 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290586 entries, 0 to 290585
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   user_id  290586 non-null  int64 
 1   country  290586 non-null  object
dtypes: int64(1), object(1)
memory usage: 4.4+ MB


Index(['user_id', 'country'], dtype='object')

In [ ]:
#Missing Value Check
print(ab_df.isnull().sum())
print(country_df.isnull().sum())

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64
user_id    0
country    0
dtype: int64


In [ ]:
# Duplicate User Check
ab_df["user_id"].duplicated().sum()



np.int64(3895)

In [ ]:
country_df["user_id"].duplicated().sum()

np.int64(1)

In [ ]:
#Drop duplicate user 
ab_df_cleaned = ab_df.drop_duplicates(subset="user_id")
country_df_cleaned = country_df.drop_duplicates(subset="user_id")

In [ ]:
# A/B Assignment-Page Consistency Check
pd.crosstab(ab_df_cleaned["landing_page"], ab_df_cleaned["group"])


group,control,treatment
landing_page,,
new_page,1006,144315
old_page,144226,1038


In [ ]:
# Remove Mismatch Records
ab_cleaned = ab_df_cleaned[
    ((ab_df_cleaned["group"] == "control") & (ab_df_cleaned["landing_page"] == "old_page")) |
    ((ab_df_cleaned["group"] == "treatment") & (ab_df_cleaned["landing_page"] == "new_page"))
].copy()

In [ ]:
# Merge Country Data
df_merged = ab_cleaned.merge(
    country_df,
    on="user_id",
    how="left"
)
df_merged = df_merged.drop_duplicates(subset="user_id")

In [ ]:
# Check after merge
print(df_merged.shape)
print(df_merged.isnull().sum())
print(df_merged.head())

(288541, 6)
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
country         0
dtype: int64
   user_id timestamp      group landing_page  converted country
0   851104   11:48.6    control     old_page          0      US
1   804228   01:45.2    control     old_page          0      US
2   661590   55:06.2  treatment     new_page          0      US
3   853541   28:03.1  treatment     new_page          0      US
4   864975   52:26.2    control     old_page          1      US


In [ ]:
# Final Clean Data Check
df_merged.shape
df_merged.info()
df_merged.isnull().sum()
df_merged.head()

pd.crosstab(df_merged["landing_page"], df_merged["group"])
df_merged["user_id"].duplicated().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 288541 entries, 0 to 288541
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       288541 non-null  int64 
 1   timestamp     288541 non-null  object
 2   group         288541 non-null  object
 3   landing_page  288541 non-null  object
 4   converted     288541 non-null  int64 
 5   country       288541 non-null  object
dtypes: int64(2), object(4)
memory usage: 15.4+ MB


np.int64(0)

In [ ]:
pd.crosstab(df_merged["landing_page"], df_merged["group"])


group,control,treatment
landing_page,,
new_page,0,144315
old_page,144226,0


In [ ]:
# =========================
# Time variable processing
# =========================

# Keep the original time values for reference
df_merged["time_raw"] = df_merged["timestamp"].astype(str).str.strip()

# Extract minute and second from MM:SS.s format
time_parts = df_merged["time_raw"].str.extract(
    r"^(?P<minute>\d{1,2}):(?P<second>\d{1,2}(?:\.\d+)?)$"
)

# Convert extracted values to numeric type
df_merged["minute"] = pd.to_numeric(time_parts["minute"], errors="coerce")
df_merged["second"] = pd.to_numeric(time_parts["second"], errors="coerce")

# Convert time into total elapsed seconds and minutes
df_merged["elapsed_seconds"] = df_merged["minute"] * 60 + df_merged["second"]
df_merged["elapsed_minutes"] = df_merged["elapsed_seconds"] / 60

# Create four elapsed-time buckets: 0-15, 15-30, 30-45, and 45-60 minutes
bins = [0, 15, 30, 45, 60.0001]
labels = ["0-15_min", "15-30_min", "30-45_min", "45-60_min"]

df_merged["time_bucket"] = pd.cut(
    df_merged["elapsed_minutes"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# Check the processed time columns
df_merged[["time_raw", "minute", "second", "elapsed_minutes", "time_bucket"]].head(20)

,time_raw,minute,second,elapsed_minutes,time_bucket
0,11:48.6,11,48.6,11.810000,0-15_min
1,01:45.2,1,45.2,1.753333,0-15_min
2,55:06.2,55,6.2,55.103333,45-60_min
3,28:03.1,28,3.1,28.051667,15-30_min
4,52:26.2,52,26.2,52.436667,45-60_min
5,20:49.1,20,49.1,20.818333,15-30_min
6,26:46.9,26,46.9,26.781667,15-30_min
7,48:29.5,48,29.5,48.491667,45-60_min
8,58:09.0,58,9.0,58.150000,45-60_min
9,11:06.6,11,6.6,11.110000,0-15_min


In [ ]:
# =========================
# Drop intermediate time-processing columns
# =========================

cols_to_drop = [
    "minute",
    "second",
    "elapsed_seconds"
]

df_merged = df_merged.drop(
    columns=[col for col in cols_to_drop if col in df_merged.columns]
)

df_merged.head()

,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


In [ ]:
df_merged["time_bucket"].value_counts().sort_index()

time_bucket
0-15_min     71725
15-30_min    72264
30-45_min    72272
45-60_min    72280
Name: count, dtype: int64

In [ ]:
from src.config import CLEANED_DATA_PATH, PROCESSED_DIR

# Create the processed data directory if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
df_merged.to_csv(CLEANED_DATA_PATH, index=False)

print("Saved cleaned dataset to:", CLEANED_DATA_PATH)

Saved cleaned dataset to: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
